# F_S3 — Shape Classification on New Parameterized Data

Implements the full **global-to-shape decision hierarchy**:

1. **Gate 1** — MIC → global relationship strength (strong / intermediate / none)
2. **Gate 2** — |Pearson| & |Spearman| → Simple vs Complex
3. **Simple path, power gate** — power-law fit → Linear / Concave / Convex / next cubic–S–Threshold gate
4. **Complex path** — Branching / Threshold / Turning-point → subtypes

Reads from `F/output/S1_parameterized/` and `F/output/S2_parameterized/`.

In [1]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings('ignore')

def count_percent(count, total):
    count = int(count)
    total = int(total)
    return f'{count:,} ({count / total:.1%})' if total else f'{count:,} (N/A)'

def locate_repo_root() -> Path:
    for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (root / 'D').is_dir() and (root / 'E').is_dir():
            return root
    raise FileNotFoundError('Could not locate repo root.')

REPO_ROOT = locate_repo_root()
S1_DIR = REPO_ROOT / 'F' / 'output' / 'S1_parameterized'
S2_DIR = REPO_ROOT / 'F' / 'output' / 'S2_metrics'
OUT_DIR = REPO_ROOT / 'F' / 'output' / 'S3_shape_classification_new_data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Gate 1 thresholds ──
MIC_STRONG = 0.8
MIC_INTERMEDIATE = 0.6

# ── Gate 2 thresholds ──
SIMPLE_CORR_THRESHOLD = 0.7

# ── Simple path thresholds ──
R2_POWER_THRESHOLD = 0.90          # min R² for adequate power-law fit
LINEAR_B_TOLERANCE = 0.05          # |b - 1| <= this → Linear
S_SHAPE_SHARPNESS_MAX = 6.0        # max deriv / median deriv below this → S-shaped (not threshold)
THRESHOLD_SHARPNESS_MIN = 6.0      # max deriv / median deriv above this → Threshold

# ── Complex path thresholds ──
BRANCHING_GAP_RATIO = 0.25         # gap / y_range in a bin to count as branching evidence
BRANCHING_MIN_BINS = 3             # min bins showing gaps
COMPLEX_THRESHOLD_DERIV_RATIO = 5.0  # derivative spike ratio for threshold detection

print(f'Gate 1: strong MIC >= {MIC_STRONG}, intermediate MIC >= {MIC_INTERMEDIATE}')
print(f'Gate 2: Simple when |r| >= {SIMPLE_CORR_THRESHOLD} AND |ρ| >= {SIMPLE_CORR_THRESHOLD}')
print(f'Simple path: power-law R² >= {R2_POWER_THRESHOLD}, linear |b−1| <= {LINEAR_B_TOLERANCE}')
print(f'Complex path: branching gap ratio >= {BRANCHING_GAP_RATIO} in >= {BRANCHING_MIN_BINS} bins')

Gate 1: strong MIC >= 0.8, intermediate MIC >= 0.6
Gate 2: Simple when |r| >= 0.7 AND |ρ| >= 0.7
Simple path: power-law R² >= 0.9, linear |b−1| <= 0.05
Complex path: branching gap ratio >= 0.25 in >= 3 bins


/Users/mimi/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Family catalogue

Ground-truth properties for each signal family, used for validation at the end.

In [2]:
# Family catalogue: ground-truth properties for each signal family.
# F17 (Quadratic peak) was removed in F; F18 is standalone.
FAMILY_CATALOGUE = {
    'F01': {'name': 'Linear positive',           'direction': 'positive',     'linearity': 'linear',              'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'none',                  'tp': 0},
    'F02': {'name': 'Linear negative',           'direction': 'negative',     'linearity': 'linear',              'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'none',                  'tp': 0},
    'F03': {'name': 'Power convex positive',     'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F04': {'name': 'Power convex negative',     'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F05': {'name': 'Power concave positive',    'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F06': {'name': 'Power concave negative',    'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F07': {'name': 'Saturation positive',       'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'saturation',            'tp': 0},
    'F08': {'name': 'Saturation negative',       'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'saturation',            'tp': 0},
    'F09': {'name': 'Log positive',              'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F10': {'name': 'Log negative',              'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F11': {'name': 'Exponential positive',      'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'convex', 'special_shape': 'none',                  'tp': 0},
    'F12': {'name': 'Exponential negative',      'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'concave','special_shape': 'none',                  'tp': 0},
    'F13': {'name': 'S-curve positive',          'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'mixed',  'special_shape': 'S-curve',               'tp': 0},
    'F14': {'name': 'S-curve negative',          'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'mixed',  'special_shape': 'S-curve',               'tp': 0},
    'F15': {'name': 'Threshold positive',        'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'threshold',             'tp': 0},
    'F16': {'name': 'Threshold negative',        'direction': 'negative',     'linearity': 'nonlinear',           'monotonicity': 'monotonic',     'curvature': 'N/A',    'special_shape': 'threshold',             'tp': 0},
    'F18': {'name': 'Quadratic valley (U)',       'direction': 'none/local',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'convex', 'special_shape': 'valley',                'tp': 1},
    'F19': {'name': 'Spike',                     'direction': 'none/local',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'N/A',    'special_shape': 'spike',                 'tp': 1},
    'F20': {'name': 'Inverted Spike',            'direction': 'none/local',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'N/A',    'special_shape': 'inverted spike',        'tp': 1},
    'F21': {'name': 'Cubic',                     'direction': 'none/mixed',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'mixed',  'special_shape': 'cubic',                 'tp': 2},
    'F22': {'name': 'Oscillation',               'direction': 'none/mixed',   'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'mixed',  'special_shape': 'oscillation',           'tp': 'multiple'},
    'F23': {'name': 'Two Lines',                 'direction': 'positive',     'linearity': 'multi-branch linear', 'monotonicity': 'multi-branch',  'curvature': 'N/A',    'special_shape': 'two lines',             'tp': 0},
    'F24': {'name': 'Line and Parabola',         'direction': 'none/local',   'linearity': 'multi-branch',        'monotonicity': 'multi-branch',  'curvature': 'mixed',  'special_shape': 'line + parabola',       'tp': 1},
    'F25': {'name': 'Multi-regime threshold',    'direction': 'positive',     'linearity': 'multi-branch',        'monotonicity': 'multi-branch',  'curvature': 'convex', 'special_shape': 'overlapping J',         'tp': 'multiple'},
    'F26': {'name': 'Windowed threshold',        'direction': 'positive',     'linearity': 'nonlinear',           'monotonicity': 'non-monotonic', 'curvature': 'convex', 'special_shape': 'windowed J',            'tp': 'multiple'},
    'Null': {'name': 'No relationship',          'direction': 'N/A',          'linearity': 'N/A',                 'monotonicity': 'N/A',           'curvature': 'N/A',    'special_shape': 'none',                  'tp': 'N/A'},
}

# Build the expected shape_label mapping from the catalogue.
# This is the ground-truth: what shape should the classifier assign at high SNR?
EXPECTED_SHAPE = {}
for fid, props in FAMILY_CATALOGUE.items():
    if fid == 'Null':
        EXPECTED_SHAPE[fid] = 'no_global'
        continue
    d = props['direction']
    mono = props['monotonicity']
    lin = props['linearity']
    curv = props['curvature']
    shape = props['special_shape']
    tp = props['tp']

    # Simple monotonic families → gate2 = Simple
    if mono == 'monotonic':
        if lin == 'linear':
            EXPECTED_SHAPE[fid] = f'{d}_line'
        elif shape == 'S-curve':
            EXPECTED_SHAPE[fid] = f'{d}_s_shaped'
        elif shape == 'threshold':
            EXPECTED_SHAPE[fid] = f'{d}_threshold'
        elif curv == 'concave':
            EXPECTED_SHAPE[fid] = f'{d}_concave'
        elif curv == 'convex':
            EXPECTED_SHAPE[fid] = f'{d}_convex'
        else:
            EXPECTED_SHAPE[fid] = 'simple_other'
    # Non-monotonic / multi-branch → gate2 = Complex
    elif mono == 'non-monotonic':
        if tp == 1:
            EXPECTED_SHAPE[fid] = 'u_inv_u_spike'
        elif tp == 2:
            EXPECTED_SHAPE[fid] = 'complex_cubic'
        elif tp == 'multiple' and 'oscillat' in shape:
            EXPECTED_SHAPE[fid] = 'oscillatory'
        elif tp == 'multiple' and 'windowed' in shape:
            EXPECTED_SHAPE[fid] = 'multi_step_threshold'
        else:
            EXPECTED_SHAPE[fid] = 'complex_uncertain'
    elif mono == 'multi-branch':
        if 'two lines' in shape:
            EXPECTED_SHAPE[fid] = '2_way_branching'
        elif 'parabola' in shape:
            EXPECTED_SHAPE[fid] = '2_way_branching'
        elif 'overlapping' in shape or 'J' in shape:
            EXPECTED_SHAPE[fid] = 'n_way_branching'
        else:
            EXPECTED_SHAPE[fid] = 'complex_uncertain'

catalogue_df = pd.DataFrame.from_dict(FAMILY_CATALOGUE, orient='index').reset_index(names='family_id')
catalogue_df['expected_shape'] = catalogue_df['family_id'].map(EXPECTED_SHAPE)
print(f'{len(FAMILY_CATALOGUE)} families loaded')
display(catalogue_df[['family_id', 'name', 'monotonicity', 'curvature', 'special_shape', 'expected_shape']])

26 families loaded


,family_id,name,monotonicity,curvature,special_shape,expected_shape
0,F01,Linear positive,monotonic,N/A,none,positive_line
1,F02,Linear negative,monotonic,N/A,none,negative_line
2,F03,Power convex positive,monotonic,convex,none,positive_convex
3,F04,Power convex negative,monotonic,concave,none,negative_concave
4,F05,Power concave positive,monotonic,concave,none,positive_concave
5,F06,Power concave negative,monotonic,convex,none,negative_convex
6,F07,Saturation positive,monotonic,concave,saturation,positive_concave
7,F08,Saturation negative,monotonic,convex,saturation,negative_convex
8,F09,Log positive,monotonic,concave,none,positive_concave
9,F10,Log negative,monotonic,convex,none,negative_convex


## Load data

In [3]:
cases_df = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
cases_df['_pt_idx'] = np.arange(len(cases_df), dtype=np.int64)
metrics_df = pd.read_parquet(S2_DIR / 'metrics_full.parquet')
pts = np.load(S1_DIR / 'scatter_points.npz')
x_all = pts['x']
y_all = pts['y']
del pts

if len(x_all) != len(cases_df) or len(y_all) != len(cases_df):
    raise ValueError('Scatter arrays and cases.csv must contain the same number of rows.')
if cases_df['case_id'].duplicated().any() or metrics_df['case_id'].duplicated().any():
    raise ValueError('case_id must be unique in both cases and metrics.')

data = cases_df.merge(metrics_df, on='case_id', how='inner', validate='one_to_one')
data = data.reset_index(drop=True)
data['is_null'] = data['family_id'].eq('Null')
data['_pt_idx'] = pd.to_numeric(data['_pt_idx'], errors='raise').astype(np.int64)
if not data['_pt_idx'].between(0, len(x_all) - 1).all():
    raise IndexError('A mapped scatter-point index is outside the stored arrays.')
if data['_pt_idx'].duplicated().any():
    raise ValueError('Multiple classified cases map to the same scatter-point row.')

print(f'Loaded {len(data):,} metric-covered cases × {x_all.shape[1]} points')
print(f'  Metrics coverage: {len(data):,} / {len(cases_df):,} ({len(data) / len(cases_df):.1%})')
print(f'  Signal: {(~data["is_null"]).sum():,}   Null: {data["is_null"].sum():,}')
print(f'  MIC range: [{data["MIC"].min():.3f}, {data["MIC"].max():.3f}]')

Loaded 335,960 metric-covered cases × 500 points
  Metrics coverage: 335,960 / 1,144,080 (29.4%)
  Signal: 326,880   Null: 9,080
  MIC range: [0.122, 1.000]


## Gate 1 — Global relationship (MIC)

- **Strong**: MIC ≥ 0.8 → proceed to Gate 2
- **Intermediate**: 0.6 ≤ MIC < 0.8 → classification stops here
- **No global**: MIC < 0.6 → no relationship detected

In [4]:
def assign_gate1(mic):
    if mic >= MIC_STRONG:
        return 'strong_global'
    elif mic >= MIC_INTERMEDIATE:
        return 'intermediate_global'
    else:
        return 'no_global'

data['gate1'] = data['MIC'].apply(assign_gate1)

_g1_counts = data.groupby(['family_id', 'gate1']).size().unstack(fill_value=0)
_g1_counts = _g1_counts.reindex(
    columns=['strong_global', 'intermediate_global', 'no_global'], fill_value=0
).sort_index()
_g1_totals = _g1_counts.sum(axis=1)
_g1 = pd.DataFrame(index=_g1_counts.index)
_g1['family_name'] = _g1.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
_g1['Total'] = [count_percent(n, n) for n in _g1_totals]
for col in _g1_counts.columns:
    _g1[col] = [count_percent(n, total) for n, total in zip(_g1_counts[col], _g1_totals)]
print('=== Gate 1: Per-family MIC classification ===')
display(_g1)
print(f'\nOverall: {count_percent((data["gate1"]=="strong_global").sum(), len(data))} strong / '
      f'{count_percent((data["gate1"]=="intermediate_global").sum(), len(data))} intermediate / '
      f'{count_percent((data["gate1"]=="no_global").sum(), len(data))} no_global')

=== Gate 1: Per-family MIC classification ===


,family_name,Total,strong_global,intermediate_global,no_global
family_id,,,,,
F01,Linear positive,"27,240 (100.0%)","11,756 (43.2%)","1,881 (6.9%)","13,603 (49.9%)"
F03,Power convex positive,"27,240 (100.0%)","10,576 (38.8%)","2,346 (8.6%)","14,318 (52.6%)"
F05,Power concave positive,"27,240 (100.0%)","11,057 (40.6%)","2,064 (7.6%)","14,119 (51.8%)"
F07,Saturation positive,"27,240 (100.0%)","2,936 (10.8%)","4,487 (16.5%)","19,817 (72.7%)"
F13,S-curve positive,"27,240 (100.0%)","11,933 (43.8%)","1,860 (6.8%)","13,447 (49.4%)"
F15,Threshold positive,"27,240 (100.0%)","12,566 (46.1%)","1,447 (5.3%)","13,227 (48.6%)"
F18,Quadratic valley (U),"27,240 (100.0%)","10,290 (37.8%)","2,527 (9.3%)","14,423 (52.9%)"
F21,Cubic,"27,240 (100.0%)","6,578 (24.1%)","4,335 (15.9%)","16,327 (59.9%)"
F22,Oscillation,"27,240 (100.0%)","11,997 (44.0%)","1,627 (6.0%)","13,616 (50.0%)"



Overall: 92,462 (27.5%) strong / 32,706 (9.7%) intermediate / 210,792 (62.7%) no_global


## Gate 2 — Simple or Complex

Applied only to **strong global** cases (MIC ≥ 0.8).

- **Simple**: |Pearson r| ≥ 0.7 AND |Spearman ρ| ≥ 0.7
- **Complex**: otherwise

In [5]:
mask_strong = data['gate1'] == 'strong_global'

data['gate2'] = None
data.loc[mask_strong, 'gate2'] = np.where(
    (data.loc[mask_strong, 'pearson_r'].abs() >= SIMPLE_CORR_THRESHOLD)
    & (data.loc[mask_strong, 'spearman_rho'].abs() >= SIMPLE_CORR_THRESHOLD),
    'Simple', 'Complex'
)

_g2_data = data.loc[mask_strong]
_g2_counts = _g2_data.groupby(['family_id', 'gate2']).size().unstack(fill_value=0)
_g2_counts = _g2_counts.reindex(columns=['Simple', 'Complex'], fill_value=0).sort_index()
_g2_totals = _g2_counts.sum(axis=1)
_g2 = pd.DataFrame(index=_g2_counts.index)
_g2['family_name'] = _g2.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
_g2['Total strong'] = [count_percent(n, n) for n in _g2_totals]
for col in _g2_counts.columns:
    _g2[col] = [count_percent(n, total) for n, total in zip(_g2_counts[col], _g2_totals)]
print('=== Gate 2: Per-family Simple vs Complex (strong_global only) ===')
display(_g2)
print(f'\nOverall strong: {count_percent(_g2_data["gate2"].eq("Simple").sum(), len(_g2_data))} Simple / '
      f'{count_percent(_g2_data["gate2"].eq("Complex").sum(), len(_g2_data))} Complex')

=== Gate 2: Per-family Simple vs Complex (strong_global only) ===


,family_name,Total strong,Simple,Complex
family_id,,,,
F01,Linear positive,"11,756 (100.0%)","11,756 (100.0%)",0 (0.0%)
F03,Power convex positive,"10,576 (100.0%)","10,576 (100.0%)",0 (0.0%)
F05,Power concave positive,"11,057 (100.0%)","11,057 (100.0%)",0 (0.0%)
F07,Saturation positive,"2,936 (100.0%)","2,178 (74.2%)",758 (25.8%)
F13,S-curve positive,"11,933 (100.0%)","11,931 (100.0%)",2 (0.0%)
F15,Threshold positive,"12,566 (100.0%)","9,872 (78.6%)","2,694 (21.4%)"
F18,Quadratic valley (U),"10,290 (100.0%)","5,690 (55.3%)","4,600 (44.7%)"
F21,Cubic,"6,578 (100.0%)","3,099 (47.1%)","3,479 (52.9%)"
F22,Oscillation,"11,997 (100.0%)",0 (0.0%),"11,997 (100.0%)"



Overall strong: 66,186 (71.6%) Simple / 26,276 (28.4%) Complex


## Simple path — shape classification

Two-step cascade for **Simple** cases:

1. **Power-law fit**: y = a·x^b + c (R² ≥ 0.90)
   - |b − 1| ≤ 0.05 → **Line**
   - Otherwise a·b·(b−1) sign → **Concave** / **Convex**
2. **Cubic fit** (only when R² < 0.90):
   - ≥ 2 interior turning points → **Cubic**
   - Otherwise → **S-curve/Threshold**

In [6]:
def _power_law(x, a, b, c):
    return a * np.power(np.maximum(x, 1e-10), b) + c


def _fit_power_law(x, y):
    valid = np.isfinite(x) & np.isfinite(y) & (x > 1e-6)
    if valid.sum() < 10:
        return None
    x, y = x[valid], y[valid]
    slope = np.polyfit(x, y, 1)[0]
    a0 = slope
    for b0 in [1.0, 0.5, 2.0, 0.3, 3.0]:
        try:
            popt, _ = curve_fit(
                _power_law, x, y, p0=[a0, b0, np.median(y)],
                maxfev=5000,
                bounds=([-np.inf, 0.05, -np.inf], [np.inf, 8.0, np.inf]),
            )
            y_pred = _power_law(x, *popt)
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - y.mean()) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
            if r2 >= 0:
                return (*popt, r2)
        except (RuntimeError, ValueError):
            continue
    return None


def _fit_cubic(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 10:
        return None
    x, y = x[valid], y[valid]
    try:
        coeffs = np.polyfit(x, y, 3)
        y_pred = np.polyval(coeffs, x)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        return coeffs, r2
    except (np.RankWarning, np.linalg.LinAlgError):
        return None


def _count_turning_points(coeffs):
    """Count interior turning points of cubic in [0.05, 0.95]."""
    deriv = np.polyder(coeffs)
    roots = np.roots(deriv)
    real_roots = roots[np.isreal(roots)].real
    interior = real_roots[(real_roots >= 0.05) & (real_roots <= 0.95)]
    return len(interior), np.sort(interior)


def classify_simple_path(x, y):
    """Simple path: power-law → cubic cascade.
    Step 1: Power-law R² >= configured threshold → line / concave / convex
    Step 2: Cubic → >=2 turning points → cubic, otherwise → S-curve/threshold
    """
    # ── Step 1: Power-law ──
    pw = _fit_power_law(x, y)
    if pw is not None:
        a, b, c, r2 = pw
        if r2 >= R2_POWER_THRESHOLD:
            ab = a * b
            direction = 'positive' if ab > 0 else 'negative'
            details = {'step': 'power_law', 'a': a, 'b': b, 'c': c, 'r2': r2}
            if abs(b - 1) <= LINEAR_B_TOLERANCE:
                return f'{direction}_line', details
            curvature = a * b * (b - 1)
            if curvature < 0:
                return f'{direction}_concave', details
            else:
                return f'{direction}_convex', details

    # ── Step 2: Cubic (R² < 0.90) ──
    cb = _fit_cubic(x, y)
    if cb is None:
        return 'simple_other', {'step': 'failed'}

    coeffs, r2_cubic = cb
    n_tp, tp_locs = _count_turning_points(coeffs)
    direction_val = float(np.polyval(coeffs, 1.0) - np.polyval(coeffs, 0.0))
    direction = 'positive' if direction_val >= 0 else 'negative'
    details = {'step': 'cubic', 'coeffs': coeffs.tolist(), 'r2': r2_cubic, 'n_tp': n_tp}

    if n_tp >= 2:
        return 'cubic_two_turning_point', details

    # Otherwise → S-curve/threshold
    return f'{direction}_s_threshold', details

print('Simple path helpers defined.')

Simple path helpers defined.


## Run full classification

In [7]:
import time

pt_idx = data['_pt_idx'].values
shape_labels = np.full(len(data), '', dtype=object)
shape_details = np.full(len(data), '', dtype=object)

mask_intermediate = data['gate1'] == 'intermediate_global'
mask_no_global = data['gate1'] == 'no_global'
shape_labels[mask_intermediate.values] = 'intermediate_global'
shape_labels[mask_no_global.values] = 'no_global'
shape_details[mask_intermediate.values] = '{}'
shape_details[mask_no_global.values] = '{}'

mask_simple = (data['gate2'] == 'Simple').values
mask_complex = (data['gate2'] == 'Complex').values
idx_simple = np.where(mask_simple)[0]

print(f'To classify: {len(idx_simple):,} Simple')
print(f'Complex (not sub-classified): {mask_complex.sum():,}')
print(f'Skipped: {mask_intermediate.sum():,} intermediate + {mask_no_global.sum():,} no_global')

t0 = time.time()
for i in tqdm(idx_simple, desc='Simple path'):
    pi = pt_idx[i]
    x = x_all[pi].astype(np.float64)
    y = y_all[pi].astype(np.float64)
    label, details = classify_simple_path(x, y)
    shape_labels[i] = label
    shape_details[i] = json.dumps(details, default=str)
print(f'Simple path: {len(idx_simple):,} cases in {time.time() - t0:.1f}s')

shape_labels[mask_complex] = 'complex'
shape_details[mask_complex] = '{}'

data['shape_label'] = shape_labels
data['shape_details'] = shape_details

SHAPE_TO_BROAD = {
    'positive_line': 'line', 'negative_line': 'line',
    'positive_concave': 'concave', 'negative_concave': 'concave',
    'positive_convex': 'convex', 'negative_convex': 'convex',
    'cubic_two_turning_point': 'cubic',
    'positive_s_threshold': 'S/threshold', 'negative_s_threshold': 'S/threshold',
    'simple_other': 'other',
}
BROAD_ORDER = ['line', 'concave', 'convex', 'cubic', 'S/threshold', 'other']
data['broad_shape'] = data['shape_label'].map(SHAPE_TO_BROAD)

_simple = data.loc[mask_simple].copy()
_simple['broad_shape'] = _simple['broad_shape'].fillna('other')

# Each stage table contains only the cases that actually entered that stage.
ROUTING_FAMILIES = ['F01', 'F03', 'F05', 'F13', 'F15', 'F21']
POWER_ROUTE_ORDER = ['Linear', 'Concave', 'Convex', 'Enter cubic/S/Threshold branch']
POWER_ROUTE_EXPECTED = {
    'F01': 'Linear',
    'F03': 'Convex',
    'F05': 'Concave',
    'F13': 'Enter cubic/S/Threshold branch',
    'F15': 'Enter cubic/S/Threshold branch',
    'F21': 'Enter cubic/S/Threshold branch',
}

power_routing = _simple[_simple['family_id'].isin(ROUTING_FAMILIES)].copy()
power_routing['power_route'] = power_routing['broad_shape'].map({
    'line': 'Linear',
    'concave': 'Concave',
    'convex': 'Convex',
}).fillna('Enter cubic/S/Threshold branch')
power_routing['expected_power_route'] = power_routing['family_id'].map(POWER_ROUTE_EXPECTED)
power_routing['route_correct'] = (
    power_routing['power_route'] == power_routing['expected_power_route']
)

power_counts = power_routing.groupby(['family_id', 'power_route']).size().unstack(fill_value=0)
power_counts = power_counts.reindex(
    index=ROUTING_FAMILIES, columns=POWER_ROUTE_ORDER, fill_value=0
)
power_totals = power_counts.sum(axis=1)
power_correct = power_routing.groupby('family_id')['route_correct'].sum().reindex(
    ROUTING_FAMILIES, fill_value=0
)
power_table = pd.DataFrame({
    'Family ID': ROUTING_FAMILIES,
    'Family': [FAMILY_CATALOGUE[f]['name'] for f in ROUTING_FAMILIES],
    'Total': [f'{int(n):,}' for n in power_totals],
})
for route in POWER_ROUTE_ORDER:
    power_table[route] = [
        count_percent(power_counts.loc[f, route], power_totals.loc[f])
        for f in ROUTING_FAMILIES
    ]
power_table['Accuracy'] = [
    (
        f'{int(power_correct.loc[f]):,}/{int(power_totals.loc[f]):,} '
        f'({power_correct.loc[f] / power_totals.loc[f]:.1%})'
    ) if power_totals.loc[f] else 'N/A'
    for f in ROUTING_FAMILIES
]

power_overall_correct = int(power_routing['route_correct'].sum())
power_overall_total = len(power_routing)
power_overall = {
    'Family ID': 'Overall',
    'Family': 'All six families',
    'Total': f'{power_overall_total:,}',
    **{
        route: count_percent((power_routing['power_route'] == route).sum(), power_overall_total)
        for route in POWER_ROUTE_ORDER
    },
    'Accuracy': (
        f'{power_overall_correct:,}/{power_overall_total:,} '
        f'({power_overall_correct / power_overall_total:.1%})'
    ),
}
power_table = pd.concat([power_table, pd.DataFrame([power_overall])], ignore_index=True)

print(
    f'\n=== Power-law routing gate: R² >= {R2_POWER_THRESHOLD:.2f}, '
    f'|b−1| <= {LINEAR_B_TOLERANCE:.2f} ==='
)
display(power_table)
print(
    f'Routing accuracy: {power_overall_correct:,}/{power_overall_total:,} '
    f'({power_overall_correct / power_overall_total:.1%})'
)

# Next gate: only cases routed out of the power-law gate.
next_gate = power_routing[
    power_routing['power_route'].eq('Enter cubic/S/Threshold branch')
].copy()
NEXT_ROUTE_ORDER = ['Cubic', 'Enter S/Threshold gate', 'Other/Uncertain']
next_gate['next_route'] = next_gate['broad_shape'].map({
    'cubic': 'Cubic',
    'S/threshold': 'Enter S/Threshold gate',
    'other': 'Other/Uncertain',
}).fillna('Other/Uncertain')
NEXT_ROUTE_EXPECTED = {
    'F13': 'Enter S/Threshold gate',
    'F15': 'Enter S/Threshold gate',
    'F21': 'Cubic',
}
next_gate['expected_next_route'] = next_gate['family_id'].map(NEXT_ROUTE_EXPECTED)
next_gate['route_correct'] = next_gate['next_route'] == next_gate['expected_next_route']
next_families = [f for f in ROUTING_FAMILIES if (next_gate['family_id'] == f).any()]
next_counts = next_gate.groupby(['family_id', 'next_route']).size().unstack(fill_value=0)
next_counts = next_counts.reindex(
    index=next_families, columns=NEXT_ROUTE_ORDER, fill_value=0
)
next_totals = next_counts.sum(axis=1)
next_correct = next_gate.groupby('family_id')['route_correct'].sum().reindex(
    next_families, fill_value=0
)
next_table = pd.DataFrame({
    'Family ID': next_families,
    'Family': [FAMILY_CATALOGUE[f]['name'] for f in next_families],
    'Total entering gate': [f'{int(n):,}' for n in next_totals],
})
for route in NEXT_ROUTE_ORDER:
    next_table[route] = [
        count_percent(next_counts.loc[f, route], next_totals.loc[f])
        for f in next_families
    ]
next_table['Accuracy'] = [
    (
        f'{int(next_correct.loc[f]):,}/{int(next_totals.loc[f]):,} '
        f'({next_correct.loc[f] / next_totals.loc[f]:.1%})'
    ) if next_totals.loc[f] else 'N/A'
    for f in next_families
]

next_overall_correct = int(next_gate['route_correct'].sum())
next_overall_total = len(next_gate)
next_overall = {
    'Family ID': 'Overall',
    'Family': 'All cases entering this gate',
    'Total entering gate': f'{next_overall_total:,}',
    **{
        route: count_percent((next_gate['next_route'] == route).sum(), next_overall_total)
        for route in NEXT_ROUTE_ORDER
    },
    'Accuracy': (
        f'{next_overall_correct:,}/{next_overall_total:,} '
        f'({next_overall_correct / next_overall_total:.1%})'
    ),
}
next_table = pd.concat([next_table, pd.DataFrame([next_overall])], ignore_index=True)

print('\n=== Cubic turning-point gate (only cases entering from the previous gate) ===')
display(next_table)
print(
    f'Gate accuracy: {next_overall_correct:,}/{next_overall_total:,} '
    f'({next_overall_correct / next_overall_total:.1%})'
)

power_table.to_csv(OUT_DIR / 'power_routing_table.csv', index=False)
next_table.to_csv(OUT_DIR / 'cubic_turning_point_gate_table.csv', index=False)


To classify: 66,186 Simple
Complex (not sub-classified): 26,276
Skipped: 32,706 intermediate + 210,792 no_global


Simple path:   0%|          | 0/66186 [00:00<?, ?it/s]

Simple path:   0%|          | 101/66186 [00:00<01:05, 1003.13it/s]

Simple path:   0%|          | 211/66186 [00:00<01:02, 1058.51it/s]

Simple path:   0%|          | 330/66186 [00:00<00:59, 1115.48it/s]

...

Simple path: 66,186 cases in 125.7s

=== Power-law routing gate: R² >= 0.90, |b−1| <= 0.05 ===


,Family ID,Family,Total,Linear,Concave,Convex,Enter cubic/S/Threshold branch,Accuracy
0,F01,Linear positive,"11,756","10,290 (87.5%)",154 (1.3%),173 (1.5%),"1,139 (9.7%)","10,290/11,756 (87.5%)"
1,F03,Power convex positive,"10,576",0 (0.0%),0 (0.0%),"10,192 (96.4%)",384 (3.6%),"10,192/10,576 (96.4%)"
2,F05,Power concave positive,"11,057",0 (0.0%),"10,549 (95.4%)",0 (0.0%),508 (4.6%),"10,549/11,057 (95.4%)"
3,F13,S-curve positive,"11,931",1 (0.0%),"1,487 (12.5%)","3,292 (27.6%)","7,151 (59.9%)","7,151/11,931 (59.9%)"
4,F15,Threshold positive,"9,872",0 (0.0%),0 (0.0%),0 (0.0%),"9,872 (100.0%)","9,872/9,872 (100.0%)"
5,F21,Cubic,"3,099",0 (0.0%),845 (27.3%),772 (24.9%),"1,482 (47.8%)","1,482/3,099 (47.8%)"
6,Overall,All six families,"58,291","10,291 (17.7%)","13,035 (22.4%)","14,429 (24.8%)","20,536 (35.2%)","49,536/58,291 (85.0%)"


Routing accuracy: 49,536/58,291 (85.0%)

=== Cubic turning-point gate (only cases entering from the previous gate) ===


,Family ID,Family,Total entering gate,Cubic,Enter S/Threshold gate,Other/Uncertain,Accuracy
0,F01,Linear positive,"1,139",0 (0.0%),"1,139 (100.0%)",0 (0.0%),"0/1,139 (0.0%)"
1,F03,Power convex positive,384,0 (0.0%),384 (100.0%),0 (0.0%),0/384 (0.0%)
2,F05,Power concave positive,508,0 (0.0%),508 (100.0%),0 (0.0%),0/508 (0.0%)
3,F13,S-curve positive,"7,151","2,084 (29.1%)","5,067 (70.9%)",0 (0.0%),"5,067/7,151 (70.9%)"
4,F15,Threshold positive,"9,872","5,100 (51.7%)","4,772 (48.3%)",0 (0.0%),"4,772/9,872 (48.3%)"
5,F21,Cubic,"1,482","1,482 (100.0%)",0 (0.0%),0 (0.0%),"1,482/1,482 (100.0%)"
6,Overall,All cases entering this gate,"20,536","8,666 (42.2%)","11,870 (57.8%)",0 (0.0%),"11,321/20,536 (55.1%)"


Gate accuracy: 11,321/20,536 (55.1%)


## Save results

In [8]:
output_cols = [
    'case_id', '_pt_idx', 'family_id', 'family_name', 'canonical_family',
    'strength_name', 'snr', 'snr_float', 'replicate',
    'MIC', 'pearson_r', 'spearman_rho',
    'gate1', 'gate2', 'shape_label', 'broad_shape', 'shape_details',
]
if 'is_representative' in data.columns:
    output_cols.insert(4, 'is_representative')

result = data[output_cols].copy()
result = result.rename(columns={'_pt_idx': 'scatter_point_index'})
result.to_parquet(OUT_DIR / 'shape_classification.parquet', index=False)
result.to_csv(OUT_DIR / 'shape_classification.csv', index=False)

print(f'Saved {len(result):,} rows to {OUT_DIR.relative_to(REPO_ROOT)}/')
print(f'  shape_classification.parquet')
print(f'  shape_classification.csv')

Saved 335,960 rows to F/output/S3_shape_classification_new_data/
  shape_classification.parquet
  shape_classification.csv


## Summary & validation

Compare predicted shape labels against the known `family_id` ground truth.

In [9]:
# Overall shape label distribution
print('=== Shape label distribution ===')
_shape_counts = data['shape_label'].value_counts()
_shape_distribution = pd.DataFrame({
    'shape_label': _shape_counts.index,
    'Count (share)': [count_percent(n, len(data)) for n in _shape_counts],
})
print(_shape_distribution.to_string(index=False))
print()

# Evaluate the current Simple-path label resolution rather than comparing it
# with finer Complex labels that this notebook does not attempt to predict.
EXPECTED_BROAD_SHAPE = {
    'F01': 'line', 'F02': 'line',
    'F03': 'convex', 'F04': 'concave',
    'F05': 'concave', 'F06': 'convex',
    'F07': 'concave', 'F08': 'convex',
    'F09': 'concave', 'F10': 'convex',
    'F11': 'convex', 'F12': 'concave',
    'F13': 'S/threshold', 'F14': 'S/threshold',
    'F15': 'S/threshold', 'F16': 'S/threshold',
    'F21': 'cubic',
}

high_snr = data[pd.to_numeric(data['snr_float'], errors='coerce') >= 10].copy()
high_snr['expected_broad_shape'] = high_snr['family_id'].map(EXPECTED_BROAD_SHAPE)
eligible_high = high_snr[high_snr['expected_broad_shape'].notna()].copy()
eligible_high['entered_simple'] = eligible_high['gate2'].eq('Simple')
shape_eval = eligible_high[eligible_high['entered_simple']].copy()
shape_eval['predicted_broad_shape'] = shape_eval['broad_shape'].fillna('other')
shape_eval['correct'] = (
    shape_eval['predicted_broad_shape'] == shape_eval['expected_broad_shape']
)

coverage = eligible_high.groupby('family_id', as_index=False).agg(
    family_name=('family_name', 'first'),
    expected_broad_shape=('expected_broad_shape', 'first'),
    high_snr_input=('case_id', 'size'),
    entered_simple=('entered_simple', 'sum'),
)
coverage['simple_path_coverage'] = coverage['entered_simple'] / coverage['high_snr_input']
shape_accuracy = shape_eval.groupby('family_id', as_index=False).agg(
    correctly_classified=('correct', 'sum'),
    top_predicted=('predicted_broad_shape', lambda s: s.value_counts().index[0]),
)
accuracy = coverage.merge(shape_accuracy, on='family_id', how='left')
accuracy['correctly_classified'] = accuracy['correctly_classified'].fillna(0).astype(int)
accuracy['conditional_accuracy'] = np.divide(
    accuracy['correctly_classified'], accuracy['entered_simple'],
    out=np.full(len(accuracy), np.nan), where=accuracy['entered_simple'].to_numpy() > 0,
)

print('=== High-SNR Simple-path coverage and broad-shape accuracy ===')
accuracy_display = accuracy[[
    'family_id', 'family_name', 'expected_broad_shape', 'high_snr_input',
    'entered_simple', 'correctly_classified', 'top_predicted',
]].copy()
accuracy_display['High-SNR input'] = [count_percent(n, n) for n in accuracy['high_snr_input']]
accuracy_display['Entered Simple'] = [
    count_percent(n, total) for n, total in zip(accuracy['entered_simple'], accuracy['high_snr_input'])
]
accuracy_display['Correct'] = [
    count_percent(n, total) for n, total in zip(accuracy['correctly_classified'], accuracy['entered_simple'])
]
accuracy_display = accuracy_display[[
    'family_id', 'family_name', 'expected_broad_shape', 'High-SNR input',
    'Entered Simple', 'top_predicted', 'Correct',
]]
print(accuracy_display.to_string(index=False))
overall_coverage = len(shape_eval) / len(eligible_high) if len(eligible_high) else np.nan
overall_accuracy = shape_eval['correct'].mean() if len(shape_eval) else np.nan
print(f'\nSimple-path coverage: {len(shape_eval):,}/{len(eligible_high):,} ({overall_coverage:.1%})')
print(f'Conditional broad-shape accuracy: {shape_eval["correct"].sum():,}/{len(shape_eval):,} ({overall_accuracy:.1%})')
print('Complex families are excluded here because this notebook currently reports only the broad label Complex.')
accuracy.to_csv(OUT_DIR / 'simple_path_broad_accuracy_high_snr.csv', index=False)

=== Shape label distribution ===
            shape_label   Count (share)
              no_global 210,792 (62.7%)
    intermediate_global   32,706 (9.7%)
                complex   26,276 (7.8%)
        positive_convex   16,738 (5.0%)
       positive_concave   13,601 (4.0%)
   positive_s_threshold   11,915 (3.5%)
          positive_line   10,291 (3.1%)
cubic_two_turning_point    9,433 (2.8%)
   negative_s_threshold    3,219 (1.0%)
       negative_concave      845 (0.3%)
        negative_convex      144 (0.0%)



=== High-SNR Simple-path coverage and broad-shape accuracy ===
family_id            family_name expected_broad_shape  High-SNR input  Entered Simple top_predicted         Correct
      F01        Linear positive                 line 10,440 (100.0%) 10,440 (100.0%)          line  10,150 (97.2%)
      F03  Power convex positive               convex 10,440 (100.0%)  10,094 (96.7%)        convex 10,089 (100.0%)
      F05 Power concave positive              concave 10,440 (100.0%)  10,404 (99.7%)       concave  10,395 (99.9%)
      F07    Saturation positive              concave 10,440 (100.0%)   2,178 (20.9%)       concave   1,411 (64.8%)
      F13       S-curve positive          S/threshold 10,440 (100.0%) 10,440 (100.0%)   S/threshold   3,989 (38.2%)
      F15     Threshold positive          S/threshold 10,440 (100.0%)   8,264 (79.2%)         cubic   4,117 (49.8%)
      F21                  Cubic                cubic 10,440 (100.0%)   3,099 (29.7%)         cubic   1,482 (47.8%)

Simple-p

In [10]:
# Cross-tabulation: family × shape_label for high SNR
print('=== Family × Shape Label (SNR >= 10, strong_global only) ===')
ct_counts = high_snr[high_snr['gate1'] == 'strong_global'].groupby(
    ['family_id', 'shape_label']
).size().unstack(fill_value=0)
ct_totals = ct_counts.sum(axis=1)
ct = pd.DataFrame(index=ct_counts.index)
ct['family_name'] = ct.index.map(lambda f: FAMILY_CATALOGUE.get(f, {}).get('name', ''))
ct['Total'] = [count_percent(n, n) for n in ct_totals]
for col in ct_counts.columns:
    ct[col] = [count_percent(n, total) for n, total in zip(ct_counts[col], ct_totals)]
display(ct)

=== Family × Shape Label (SNR >= 10, strong_global only) ===


,family_name,Total,complex,cubic_two_turning_point,negative_concave,negative_convex,negative_s_threshold,positive_concave,positive_convex,positive_line,positive_s_threshold
family_id,,,,,,,,,,,
F01,Linear positive,"10,440 (100.0%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),134 (1.3%),151 (1.4%),"10,150 (97.2%)",5 (0.0%)
F03,Power convex positive,"10,094 (100.0%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),"10,089 (100.0%)",0 (0.0%),5 (0.0%)
F05,Power concave positive,"10,404 (100.0%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),"10,395 (99.9%)",0 (0.0%),0 (0.0%),9 (0.1%)
F07,Saturation positive,"2,936 (100.0%)",758 (25.8%),767 (26.1%),0 (0.0%),0 (0.0%),0 (0.0%),"1,411 (48.1%)",0 (0.0%),0 (0.0%),0 (0.0%)
F13,S-curve positive,"10,440 (100.0%)",0 (0.0%),"1,671 (16.0%)",0 (0.0%),0 (0.0%),0 (0.0%),"1,487 (14.2%)","3,292 (31.5%)",1 (0.0%),"3,989 (38.2%)"
F15,Threshold positive,"10,436 (100.0%)","2,172 (20.8%)","4,147 (39.7%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),"4,117 (39.4%)"
F18,Quadratic valley (U),"9,992 (100.0%)","4,318 (43.2%)",0 (0.0%),0 (0.0%),107 (1.1%),"3,212 (32.1%)",0 (0.0%),"2,345 (23.5%)",0 (0.0%),10 (0.1%)
F21,Cubic,"6,578 (100.0%)","3,479 (52.9%)","1,482 (22.5%)",845 (12.8%),37 (0.6%),0 (0.0%),0 (0.0%),735 (11.2%),0 (0.0%),0 (0.0%)
F22,Oscillation,"10,440 (100.0%)","10,440 (100.0%)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
